# Autonomous Incident Response System for AWS
---

In this example, we will build an Agentic system to respond to incidents in your AWS accounts. This is a multi-agent system that composes 4 main components: 

1. **Monitoring**: This composes of a couple of aspects which includes monitoring CloudWatch alarms on the pre-built alarms that have already been set in your account. This might include high `CPU` usage, unhealthy load balancers, SageMaker instance cost allocations, etc. This would also include observing logs from different services from your account and classifying those logs into `Critical` (for example service down, `CPU`>`90%`), `Warning` (for example, latency > threshold, or if something goes beyond a threshold for a specific service) and `Informational` (for example, routine backups, information on various running applications in the AWS account, etc.).

1. **Diagnosis**: This includes diagnosis events that are seen through the monitoring agent. This can include querying `AWS` CloudTrail for additional data, X-Ray data and document these findings in reports that can be saved and used later in the resolution process. This would contain information only on the errors and the different services that need a resolution.

1. **Resolution**: This portion of the solution will be triggered by a diagnosis done from the step before. Once the diagnoses is done with the clear report, then this portion starts to remediate certain actions, such as adjusting EC2 auto-scaling group capacities, invoking functions to rollback deployments, etc. This agent is an essential part of the system since it will be using AWS `API`s in real time to manage the resources.

1. **Communication**: Last, this agent is responsible for keeping track of updates, creating and updating tickets in Jira, sending real time notifications to Slack with the incident details and resolution updates.

This solution will also contain aspects for observabilitiy and tracing but without further ado, let's get right into it.

In [1]:
# LangGraph is a low level orchestration framework for building controllable agents. 
# While langchain provides integrations and composable components to streamline LLM application development, 
# the LangGraph library enables agent orchestration, long term memory, human in the loop and customizable architectures.

In [2]:
import boto3
import logging
from datetime import datetime, timedelta
from typing import Annotated, List, Dict, Any
from typing_extensions import TypedDict
# import langgraph relevant libraries
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Import the memory saver to save in checkpoint and in some thread to retain agent's memory
from langgraph.checkpoint.memory import MemorySaver

# langchain imports
from langchain_aws.chat_models import ChatBedrockConverse
from langchain_core.tools import tool

In [3]:
import os
from colorama import init, Fore, Style
import logging

# Initialize colorama
init()

# Create a custom formatter
class ColoredFormatter(logging.Formatter):
    def format(self, record):
        message = record.msg
        if isinstance(message, list):
            # Process each message in the list
            formatted_messages = []
            for msg in message:
                if msg.__class__.__name__ == 'HumanMessage':
                    formatted_msg = f"{Fore.GREEN}[Human] {msg.content}{Style.RESET_ALL}"
                elif msg.__class__.__name__ == 'AIMessage':
                    formatted_msg = f"{Fore.BLUE}[AI] {msg.content}{Style.RESET_ALL}"
                elif msg.__class__.__name__ == 'ToolMessage':
                    formatted_msg = f"{Fore.YELLOW}[Tool] {msg.content}...{Style.RESET_ALL}"
                else:
                    formatted_msg = str(msg)
                formatted_messages.append(formatted_msg)
            record.msg = '\n'.join(formatted_messages)
        return super().format(record)
# Set up logger with the custom formatter
logger = logging.getLogger(__name__)
handler = logging.StreamHandler()
handler.setFormatter(ColoredFormatter('%(message)s'))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

In [4]:
# define the constants
AMAZON_NOVA_PRO_MODEL_ID: str = 'us.amazon.nova-pro-v1:0'

### State definitions
---

First, we will define the state for our sub agents: for Monitoring, Diagnosis, Remediation and the Supervisor. Since all of these will have similar states, let's go ahead and define a common `incidentState`.

In [7]:
from typing import List, Optional

class IncidentState(TypedDict):
    """
    A TypedDict class representing the state of monitoring.
    """
    # This tracks the user messages in the history and will be 
    # used to check for which is the next node/sub agent to use in this
    # agentic architecture
    messages: List[Dict]
    # This contains information on the alarm statuses in the AWS account
    alarms: Optional[List[Dict]]
    # This contains information on the metrics in the AWS account
    metrics: Optional[Dict]
    # Instance ids of your EC2 instances
    instance_ids: Optional[List[str]]
    # This contains information on the diagnosis that can be done to remediate
    diagnosis_report: Optional[str]
    remediation_actions: Optional[List[str]]
    notification_status: Optional[str]

With the help of this unified state, this does as follows:

1. **Ensures consistency**: Each agent in this case works with a consistent structure.

1. **Ease of communication**: This facilitates simpler data passing between nodes.

1. **Traceability**: Incident lifecycle remains centralized.

In [8]:
from langchain_core.messages import HumanMessage

llm = ChatBedrockConverse(
    model_id = AMAZON_NOVA_PRO_MODEL_ID, 
    temperature = 0.1,
)

#### Define monitoring tools
---

In [9]:
import boto3
from datetime import timedelta
from langchain_core.tools import BaseTool, tool
from langgraph.prebuilt import create_react_agent

# define some boto3 and AWS clients
cloudwatch_client = boto3.client('cloudwatch')
cloudtrail_client = boto3.client('cloudtrail')
xray_client = boto3.client('xray')
autoscaling_client = boto3.client('autoscaling')
ec2_client = boto3.client('ec2')

@tool
def fetch_ec2_metrics_for_alarm_instances(
    alarm_state: Annotated[str, "State of alarms to check, e.g., 'ALARM' or 'OK'"] = "ALARM",
    period: int = 300
) -> Optional[Dict[str, Dict[str, Optional[float]]]]:
    """
    Fetches alarms once and retrieves key utilization metrics for affected EC2 instances.
    Returns a dictionary mapping instance IDs to their metrics summary.
    """
    try:
        # This gets the alarm response from the cloudwatch client. This in this example can be EC2
        # instances that cross certain thresholds in your AWS account
        alarm_response = cloudwatch_client.describe_alarms(StateValue=alarm_state, MaxRecords=100)
    except Exception as e:
        print(f"Error fetching alarms: {e}")
        return None
    if not alarm_response or not alarm_response.get("MetricAlarms"):
        print(f"No alarms found in state: {alarm_state}")
        return {}
    # Identify affected EC2 instance IDs from alarms
    alarm_instances = {dim["Value"]
                       for alarm in alarm_response["MetricAlarms"]
                       for dim in alarm.get("Dimensions", [])
                       if dim["Name"] == "InstanceId"}
    # metrics of interest to fetch, using these metrics we will be able to notify the 
    # user via slack or create a ticket on jira for them to view and track
    metrics_to_fetch = [
        "CPUUtilization", "NetworkIn", "NetworkOut",
        "DiskReadOps", "DiskWriteOps", "StatusCheckFailed"
    ]
    instance_metrics_summary = {}
    # Fetch metrics for each instance
    for instance_id in alarm_instances:
        instance_summary = {}
        for metric_name in metrics_to_fetch:
            try:
                response = cloudwatch_client.get_metric_statistics(
                    Namespace="AWS/EC2",
                    MetricName=metric_name,
                    Dimensions=[{'Name': 'InstanceId', 'Value': instance_id}],
                    Period=period,
                    Statistics=['Average'],
                    StartTime=datetime.utcnow() - timedelta(minutes=10),
                    EndTime=datetime.utcnow()
                )
                datapoints = response.get("Datapoints", [])
                if datapoints:
                    # Sort datapoints by timestamp to pick latest
                    datapoints.sort(key=lambda x: x['Timestamp'], reverse=True)
                    instance_summary[metric_name] = datapoints[0]["Average"]
                else:
                    instance_summary[metric_name] = None
            except Exception as e:
                print(f"Error fetching metric '{metric_name}' for {instance_id}: {e}")
                instance_summary[metric_name] = None
        instance_metrics_summary[instance_id] = instance_summary
    return instance_metrics_summary

monitoring_toolkit = [fetch_ec2_metrics_for_alarm_instances]

In [10]:
# create the monitoring agent
monitoring_agent = create_react_agent(llm, tools=monitoring_toolkit, prompt="This agent is used to monitor AWS alarms and classify incidents within it as Critical, Warning or Informational.")
logger.info(f"Created the monitoring agent: {monitoring_agent}")

Created the monitoring agent: <langgraph.graph.state.CompiledStateGraph object at 0x10ea7ee40>


In [12]:
# Create the node for the monitoring agent
def monitoring_node(state: IncidentState):
    result = monitoring_agent.invoke(state)
    updated_state = {
        "messages": state["messages"] + [{"content": result["messages"][-1].content, "role": "monitoring"}],
        "alarms": result.get("alarms"),
        "metrics": result.get("metrics"),
        "instance_ids": result.get("instance_ids")
    }
    return updated_state

#### Define diagnosis tools
---

In [16]:
@tool
def get_ec2_instance_details(instance_ids: List[str]) -> Dict:
    """
    Fetches detailed information about EC2 instances by their IDs.
    """
    response = ec2_client.describe_instances(InstanceIds=instance_ids)
    
    instance_details = {}
    for reservation in response.get('Reservations', []):
        for instance in reservation.get('Instances', []):
            instance_id = instance['InstanceId']
            instance_details[instance_id] = {
                'InstanceType': instance.get('InstanceType'),
                'State': instance.get('State', {}).get('Name'),
                'LaunchTime': instance.get('LaunchTime'),
                'PrivateIpAddress': instance.get('PrivateIpAddress'),
                'PublicIpAddress': instance.get('PublicIpAddress'),
                'VpcId': instance.get('VpcId'),
                'SubnetId': instance.get('SubnetId'),
                'SecurityGroups': [sg.get('GroupId') for sg in instance.get('SecurityGroups', [])]
            }
    return instance_details

@tool
def check_instance_status_checks(instance_ids: List[str]) -> Dict:
    """
    Fetches status check results for EC2 instances.
    """
    response = ec2_client.describe_instance_status(InstanceIds=instance_ids)
    
    status_checks = {}
    for status in response.get('InstanceStatuses', []):
        instance_id = status['InstanceId']
        status_checks[instance_id] = {
            'InstanceStatus': status.get('InstanceStatus', {}).get('Status'),
            'SystemStatus': status.get('SystemStatus', {}).get('Status'),
            'Events': [event.get('Description') for event in status.get('Events', [])]
        }
    
    return status_checks

@tool
def get_autoscaling_activity(instance_ids: List[str]) -> Dict:
    """
    Checks if the instances are part of an Auto Scaling group and fetches related activity.
    """
    # First, check which ASGs these instances belong to
    instance_to_asg = {}
    
    try:
        asg_response = autoscaling_client.describe_auto_scaling_instances(InstanceIds=instance_ids)
        for instance in asg_response.get('AutoScalingInstances', []):
            instance_to_asg[instance['InstanceId']] = instance['AutoScalingGroupName']
    except Exception as e:
        return {"error": f"Failed to fetch Auto Scaling groups: {str(e)}"}
    
    # For each ASG, get recent activities
    asg_activities = {}
    for asg_name in set(instance_to_asg.values()):
        try:
            activities = autoscaling_client.describe_scaling_activities(
                AutoScalingGroupName=asg_name,
                MaxRecords=10
            )
            asg_activities[asg_name] = [
                {
                    'ActivityId': activity.get('ActivityId'),
                    'Description': activity.get('Description'),
                    'StatusCode': activity.get('StatusCode'),
                    'StatusMessage': activity.get('StatusMessage'),
                    'StartTime': activity.get('StartTime'),
                    'EndTime': activity.get('EndTime'),
                    'Cause': activity.get('Cause')
                }
                for activity in activities.get('Activities', [])
            ]
        except Exception as e:
            asg_activities[asg_name] = {"error": f"Failed to fetch activities: {str(e)}"}
    
    # Map results back to instances
    result = {}
    for instance_id in instance_ids:
        asg_name = instance_to_asg.get(instance_id)
        if asg_name:
            result[instance_id] = {
                'AutoScalingGroupName': asg_name,
                'Activities': asg_activities.get(asg_name, [])
            }
        else:
            result[instance_id] = {'AutoScalingGroupName': None}
    
    return result

@tool
def get_recent_cloudtrail_for_instances(instance_ids: List[str], minutes: int = 60) -> Dict:
    """
    Fetches CloudTrail events related to specific EC2 instances.
    """
    results = {}
    
    for instance_id in instance_ids:
        try:
            events = cloudtrail_client.lookup_events(
                LookupAttributes=[
                    {
                        'AttributeKey': 'ResourceName',
                        'AttributeValue': instance_id
                    },
                ],
                StartTime=datetime.utcnow() - timedelta(minutes=minutes),
                EndTime=datetime.utcnow(),
                MaxResults=20
            )
            
            results[instance_id] = [
                {
                    'EventName': event.get('EventName'),
                    'EventTime': event.get('EventTime'),
                    'Username': event.get('Username'),
                    'ResourceName': event.get('Resources', [{}])[0].get('ResourceName') if event.get('Resources') else None
                }
                for event in events.get('Events', [])
            ]
        except Exception as e:
            results[instance_id] = {"error": f"Failed to fetch CloudTrail events: {str(e)}"}
    
    return results

# Create the expanded diagnosis toolkit with the new EC2-focused tools
diagnosis_toolkit = [
    get_ec2_instance_details,
    check_instance_status_checks,
    get_autoscaling_activity,
    get_recent_cloudtrail_for_instances
]

In [17]:
# Create the diagnosis agent with the expanded toolkit
diagnosis_agent = create_react_agent(
    llm, 
    tools=diagnosis_toolkit, 
    prompt="""You are a specialized AWS diagnosis agent focused on analyzing EC2 instance issues.
    When given information about EC2 instances with alarms or performance issues:
    1. Gather detailed information about the instances
    2. Check instance status and recent events
    3. Look for related Auto Scaling activities
    4. Examine relevant CloudTrail logs
    5. Provide a comprehensive diagnosis with likely root causes
    6. Suggest specific remediation steps based on your findings
    
    Be thorough in your analysis, but prioritize the most relevant information."""
)

In [18]:
# Define the diagnosis node for the workflow
def diagnosis_node(state: Dict):
    # Extract instance IDs from the monitoring state
    instance_ids = state.get("instance_ids", [])
    
    # If no instance IDs were identified, try to extract them from metrics
    if not instance_ids and "metrics" in state:
        instance_ids = list(state["metrics"].keys())
    
    # Create a diagnostic message using the metrics information
    diagnostic_request = f"""
    I need to diagnose issues with the following EC2 instances that have triggered alarms:
    {', '.join(instance_ids)}
    
    The current metrics for these instances are:
    {state.get('metrics', {})}
    
    Please analyze these instances to determine:
    1. What is causing the current alarms or performance issues
    2. Whether this is likely an infrastructure problem, application issue, or capacity constraint
    3. What specific actions should be taken to resolve the issues
    """
    
    # Invoke the diagnosis agent
    result = diagnosis_agent.invoke({
        "messages": [HumanMessage(content=diagnostic_request)],
        "instance_ids": instance_ids,
        "metrics": state.get("metrics", {})
    })
    
    # Update state with diagnosis results
    updated_state = state.copy()
    updated_state["messages"] = state["messages"] + [{"content": result["messages"][-1].content, "role": "diagnosis"}]
    updated_state["diagnosis"] = {
        "analysis": result.get("analysis", {}),
        "recommendations": result.get("recommendations", [])
    }
    
    return updated_state

### Test the monitoring and diagnosis agents
---

Next, after we have defined our monitoring and diagnosis tools and created the agents, we can invoke to test how they work.

In [19]:
from langchain_core.messages import HumanMessage
content = """
I want to check the current alarms in my AWS account.
Then, summarize EC2 instance utilization metrics for me. 
"""

# Invoke monitoring agent with the given message
monitoring_response = monitoring_agent.invoke({"messages": [HumanMessage(content=content)]})
logger.info(monitoring_response["messages"])


/var/folders/jy/g9mb5j5n6c11fgdj788p5rww0000gr/T/ipykernel_66072/4156480986.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  StartTime=datetime.utcnow() - timedelta(minutes=10),
/var/folders/jy/g9mb5j5n6c11fgdj788p5rww0000gr/T/ipykernel_66072/4156480986.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  EndTime=datetime.utcnow()
[Human] 
I want to check the current alarms in my AWS account.
Then, summarize EC2 instance utilization metrics for me. 

[AI] [{'type': 'text', 'text': "<thinking> To check the current alarms in the user's AWS account and summarize EC2 instance utilization metrics, I need to use the provided tool 'fetch_ec2_metrics_for_alarm_instances'. This 

In [20]:
from langchain_core.messages import HumanMessage
content = """
My EC2 instance `i-05ed78c3e64323343` is in alarm state. Diagnose it.
"""

# Invoke monitoring agent with the given message
diagnosis_response = diagnosis_agent.invoke({"messages": [HumanMessage(content=content)]})
logger.info(diagnosis_response["messages"])


/var/folders/jy/g9mb5j5n6c11fgdj788p5rww0000gr/T/ipykernel_66072/3584380319.py:110: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  StartTime=datetime.utcnow() - timedelta(minutes=minutes),
/var/folders/jy/g9mb5j5n6c11fgdj788p5rww0000gr/T/ipykernel_66072/3584380319.py:111: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  EndTime=datetime.utcnow(),
[Human] 
My EC2 instance `i-05ed78c3e64323343` is in alarm state. Diagnose it.

[AI] [{'type': 'text', 'text': '<thinking> To diagnose the EC2 instance `i-05ed78c3e64323343` that is in an alarm state, I need to gather detailed information about the instance, check its status and recent events, look for related Auto Scaling activities, 

### Create the resolution tools

In [33]:
autoscaling_client = boto3.client('autoscaling')
ec2_client = boto3.client('ec2')
lambda_client = boto3.client('lambda')
ecs_client = boto3.client('ecs')

In [34]:
@tool
def remediate_aws_issue(
    action_type: Annotated[str, "Type of remediation action: scale, restart, modify_instance, rollback"],
    resources: List[str],
    parameters: Dict[str, Any] = {}
) -> Dict[str, Any]:
    """
    Executes a remediation action on AWS resources based on the diagnosis.
    
    Parameters:
        action_type: The type of remediation to perform (scale, restart, modify_instance, rollback)
        resources: List of resource identifiers (instance IDs, ASG names, etc.)
        parameters: Additional parameters specific to the action type:
            - For 'scale': min_size, max_size, desired_capacity
            - For 'modify_instance': instance_type
            - For 'rollback': deployment_id, version
    
    Returns:
        Dict with status and results of the remediation action
    """
    try:
        results = {}
        
        if action_type == "scale" and resources:
            # Adjust Auto Scaling group capacity
            asg_name = resources[0]
            min_size = parameters.get("min_size")
            max_size = parameters.get("max_size")
            desired_capacity = parameters.get("desired_capacity")
            
            update_params = {"AutoScalingGroupName": asg_name}
            if min_size is not None:
                update_params["MinSize"] = min_size
            if max_size is not None:
                update_params["MaxSize"] = max_size
            if desired_capacity is not None:
                update_params["DesiredCapacity"] = desired_capacity
                
            autoscaling_client.update_auto_scaling_group(**update_params)
            results["scale"] = f"Auto Scaling group '{asg_name}' updated successfully"
            
        elif action_type == "restart" and resources:
            # Restart EC2 instances
            instance_ids = resources
            response = ec2_client.reboot_instances(InstanceIds=instance_ids)
            results["restart"] = f"Reboot initiated for instances: {', '.join(instance_ids)}"
            
        elif action_type == "modify_instance" and resources:
            # Modify instance type
            instance_id = resources[0]
            instance_type = parameters.get("instance_type")
            
            if not instance_type:
                return {"status": "error", "message": "Instance type not provided"}
            
            # Check if instance is stopped
            instance_response = ec2_client.describe_instances(InstanceIds=[instance_id])
            instance_state = instance_response['Reservations'][0]['Instances'][0]['State']['Name']
            
            if instance_state != 'stopped':
                results["modify_instance"] = f"Instance {instance_id} must be stopped before changing type"
            else:
                ec2_client.modify_instance_attribute(
                    InstanceId=instance_id,
                    InstanceType={'Value': instance_type}
                )
                results["modify_instance"] = f"Instance {instance_id} type changed to {instance_type}"
                
        elif action_type == "rollback" and resources:
            # Invoke rollback function or service
            function_name = resources[0]
            
            response = lambda_client.invoke(
                FunctionName=function_name,
                InvocationType='RequestResponse',
                Payload=json.dumps(parameters)
            )
            
            response_payload = json.loads(response['Payload'].read().decode('utf-8'))
            results["rollback"] = f"Rollback initiated via Lambda {function_name}"
            
        else:
            return {"status": "error", "message": f"Invalid action type '{action_type}' or missing resources"}
            
        return {
            "status": "success",
            "message": f"{action_type} action completed successfully",
            "details": results
        }
        
    except Exception as e:
        return {"status": "error", "message": f"Failed to perform {action_type}: {str(e)}"}

# Create the resolution agent with just this one tool
resolution_toolkit = [remediate_aws_issue]

In [35]:
resolution_agent = create_react_agent(
    llm, 
    tools=resolution_toolkit, 
    prompt="""You are an AWS resolution agent responsible for remediating issues.
    Based on the diagnosis provided, select the appropriate remediation action
    and execute it with caution. Always document what action you took and why."""
)

In [37]:
content = """
My EC2 instance `i-05ed78c3e64323343` is in alarm state. Resolve it and tell me how you are doing to do so.
"""

# Invoke monitoring agent with the given message
resolution_response = resolution_agent.invoke({"messages": [HumanMessage(content=content)]})
logger.info(resolution_response["messages"])


[Human] 
My EC2 instance `i-05ed78c3e64323343` is in alarm state. Resolve it and tell me how you are doing to do so.

[AI] [{'type': 'text', 'text': '<thinking> The EC2 instance is in an alarm state, which typically indicates that there is an issue that needs to be addressed. The most common remediation actions for an EC2 instance in alarm state are to restart the instance or modify its configuration. Since the instance is already in an alarm state, restarting it might help resolve transient issues. I will proceed with restarting the instance. </thinking>\n\n'}, {'type': 'tool_use', 'name': 'remediate_aws_issue', 'input': {'action_type': 'restart', 'resources': ['i-05ed78c3e64323343']}, 'id': 'tooluse_XIUhFdDTR6uxSf9vmAn2kw'}]
[Tool] {"status": "success", "message": "restart action completed successfully", "details": {"restart": "Reboot initiated for instances: i-05ed78c3e64323343"}}...
[AI] <thinking> The restart action for the EC2 instance `i-05ed78c3e64323343` was completed successf

### Create the communication tools
---

In [42]:
def create_sns_topic(
    topic_name: str = "aws-incidents",
    display_name: str = "AWS Incident Notifications",
    subscriptions: List[Dict[str, str]] = None
) -> Dict[str, Any]:
    """
    Creates an AWS SNS topic for incident notifications and adds subscriptions.
    
    Parameters:
        topic_name: Name for the SNS topic
        display_name: Display name that appears when sending notifications
        subscriptions: List of dicts with 'protocol' and 'endpoint' keys
        
    Returns:
        Dict with status and topic ARN
    """
    try:
        # Initialize the SNS client - uses IAM role credentials
        sns = boto3.client('sns')
        
        # Create the topic
        response = sns.create_topic(
            Name=topic_name,
            Attributes={
                'DisplayName': display_name
            },
            Tags=[
                {'Key': 'Purpose', 'Value': 'Incident Notifications'},
                {'Key': 'Environment', 'Value': 'Production'}
            ]
        )
        
        topic_arn = response['TopicArn']
        
        # Add subscriptions if provided
        subscription_results = []
        if subscriptions:
            for sub in subscriptions:
                try:
                    sub_response = sns.subscribe(
                        TopicArn=topic_arn,
                        Protocol=sub['protocol'],
                        Endpoint=sub['endpoint']
                    )
                    subscription_results.append({
                        "endpoint": sub['endpoint'],
                        "protocol": sub['protocol'],
                        "status": "Success"
                    })
                except Exception as e:
                    subscription_results.append({
                        "endpoint": sub['endpoint'],
                        "protocol": sub['protocol'],
                        "status": f"Error: {str(e)}"
                    })
        
        return {
            "status": "success",
            "message": f"SNS topic '{topic_name}' created successfully",
            "topic_arn": topic_arn,
            "subscriptions": subscription_results
        }
    
    except Exception as e:
        return {
            "status": "error",
            "message": f"Error creating SNS topic: {str(e)}"
        }

In [43]:
import boto3
import json
from datetime import datetime
from typing import Dict, Any, List, Optional, Annotated

@tool
def send_aws_notification(
    summary: str,
    details: str,
    severity: Annotated[str, "Incident severity: critical, warning, or info"] = "info",
    topic_arn: str = None,
    topic_name: str = "aws-incidents",
    create_topic_if_missing: bool = True,
    email_subscriptions: List[str] = None
) -> Dict[str, Any]:
    """
    Sends an incident alert through AWS SNS, creating the topic if it doesn't exist.
    
    Parameters:
        summary: Brief summary of the incident
        details: Detailed description of the incident
        severity: Severity level (critical, warning, or info)
        topic_arn: The ARN of an existing SNS topic (if None, uses topic_name)
        topic_name: Name for the SNS topic if one needs to be created
        create_topic_if_missing: Whether to create the topic if it doesn't exist
        email_subscriptions: List of email addresses to subscribe to the topic
    
    Returns:
        Dict with status and result of the notification
    """
    try:
        # Initialize SNS client - uses IAM role credentials from the environment
        sns = boto3.client('sns')
        current_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
        
        # Check if we need to create or find a topic
        if topic_arn is None and create_topic_if_missing:
            try:
                # First try to find the topic by name
                topics_response = sns.list_topics()
                found_topic = False
                
                for topic in topics_response.get('Topics', []):
                    # Extract name from ARN and compare
                    arn_parts = topic['TopicArn'].split(':')
                    if arn_parts[-1] == topic_name:
                        topic_arn = topic['TopicArn']
                        found_topic = True
                        break
                
                # If topic not found, create it
                if not found_topic:
                    # Create the topic
                    response = sns.create_topic(
                        Name=topic_name,
                        Attributes={
                            'DisplayName': 'AWS Incident Notifications'
                        },
                        Tags=[
                            {'Key': 'Purpose', 'Value': 'Incident Notifications'},
                            {'Key': 'Environment', 'Value': 'Production'}
                        ]
                    )
                    topic_arn = response['TopicArn']
                    
                    # Add email subscriptions if provided
                    if email_subscriptions:
                        for email in email_subscriptions:
                            sns.subscribe(
                                TopicArn=topic_arn,
                                Protocol='email',
                                Endpoint=email
                            )
            except Exception as e:
                return {
                    "status": "error",
                    "message": f"Error creating or finding SNS topic: {str(e)}"
                }
        
        if not topic_arn:
            return {
                "status": "error", 
                "message": "No topic ARN provided and create_topic_if_missing is False"
            }
        
        # Prepare message
        subject = f"AWS Incident: {summary} - {severity.upper()}"
        
        message = {
            "incident_summary": summary,
            "incident_details": details,
            "severity": severity.upper(),
            "timestamp": current_time
        }
        
        # Send the notification
        response = sns.publish(
            TopicArn=topic_arn,
            Message=json.dumps(message),
            Subject=subject
        )
        
        return {
            "status": "success",
            "message": f"Alert sent via SNS",
            "message_id": response['MessageId'],
            "topic_arn": topic_arn,
            "timestamp": current_time,
            "severity": severity
        }
    
    except Exception as e:
        return {
            "status": "error",
            "message": f"Error sending AWS notification: {str(e)}"
        }

In [44]:

# Create the communication agent with the enhanced SNS tool
communication_toolkit = [send_aws_notification]
communication_agent = create_react_agent(
    llm, 
    tools=communication_toolkit, 
    prompt="""You are an AWS incident communication agent. Your job is to notify
    stakeholders about incidents through AWS SNS. Create clear, informative alerts
    that describe the issue, its impact, and the current status of resolution.
    Adjust your message tone based on severity level."""
)


In [45]:
# Define the communication node
def communication_node(state: IncidentState):
    diagnosis = state.get("diagnosis", {})
    remediation_actions = state.get("remediation_actions", [])
    instance_ids = state.get("instance_ids", [])
    
    # Determine severity based on diagnosis
    severity = "critical" if "critical" in str(diagnosis).lower() else "warning"
    
    communication_request = f"""
    I need to send a Slack alert about the following AWS incident:
    
    Affected resources: {', '.join(instance_ids) if instance_ids else 'Unknown'}
    
    Diagnosis: {diagnosis.get('analysis', 'No analysis available')}
    
    Remediation actions: {remediation_actions if remediation_actions else 'Pending'}
    
    Please create a clear Slack alert with severity {severity}.
    """
    
    result = communication_agent.invoke({
        "messages": [HumanMessage(content=communication_request)],
        "diagnosis": diagnosis,
        "remediation_actions": remediation_actions,
        "severity": severity
    })
    
    notification_content = result["messages"][-1].content
    
    updated_state = state.copy()
    updated_state["messages"] = state["messages"] + [{"content": notification_content, "role": "communication"}]
    updated_state["notification_status"] = "sent"
    
    return updated_state

In [47]:
content = """
My EC2 instance `i-05ed78c3e64323343` is in alarm state. Send a notification about it.
"""

# Invoke monitoring agent with the given message
communication_response = communication_agent.invoke({"messages": [HumanMessage(content=content)]})
logger.info(communication_response["messages"])


/var/folders/jy/g9mb5j5n6c11fgdj788p5rww0000gr/T/ipykernel_66072/1430203177.py:34: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  current_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
[Human] 
My EC2 instance `i-05ed78c3e64323343` is in alarm state. Send a notification about it.

[AI] [{'type': 'text', 'text': '<thinking> The user has reported that their EC2 instance is in an alarm state. I need to send a notification about this incident. Since the severity level was not specified, I will assume it is a "warning" by default. I will use the `send_aws_notification` tool to send the alert. </thinking>\n'}, {'type': 'tool_use', 'name': 'send_aws_notification', 'input': {'summary': 'EC2 instance i-05ed78c3e64323343 is in alarm state.', 'severity': 'warning', 'create_topic_if_missing': True, 'topic_name': 'aws-incidents', '